<a href="https://colab.research.google.com/github/haribalqis/Iman-Diary-Murobbi/blob/main/W1_NLTK_Tokenisation_Lab_(with_answers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text Processing Lab 1 - Tokenisation with NLTK**

By the end of this lab session, you should be able to:

- explain why whitespace-separated ‘words’ are not universal computational units;
- use and compare NLTK word, sentence, social-media, and regex tokenisers;
- design task-aware tokenisation rules and test them with explicit cases; and
- diagnose where a tokenizer is likely to fail across domains, scripts, and Unicode sequences.

> Complete the mini-tests to check your understanding. Several tasks have more than one defensible design; the supplied answer states one explicit specification.

In [ ]:
# Colab setup: pin the version so that tokenisation decisions are reproducible.
%pip -q install nltk==3.9.2

import re
import unicodedata
from collections import Counter

import nltk
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import (
    RegexpTokenizer,
    TreebankWordTokenizer,
    TweetTokenizer,
    WhitespaceTokenizer,
    sent_tokenize,
    word_tokenize,
    wordpunct_tokenize,
)
from nltk.tokenize.util import align_tokens

print("NLTK version:", nltk.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.2 MB/s eta 0:00:00
NLTK version: 3.9.2


> **Working rule:** tokenisation is a modelling choice, not arbitrary cleaning. State what must be preserved, implement the rule, and test the boundary decisions. NLTK’s word tokenisers produce word-like units; an LLM would normally apply its own learned subword tokenizer later.

## **Section 1 — Unicode and the whitespace baseline**

### **Example code**

In [ ]:
text = "café — 北京"
print("Python string length (code points):", len(text))
print("UTF-8 byte length:", len(text.encode("utf-8")))
print("Round trip:", text.encode("utf-8").decode("utf-8"))

message = "Great—email me at sam@example.org! #COM3110"
print("Whitespace split:", message.split())
print("Characters around the dash:", [f"{c} U+{ord(c):04X}" for c in message[5:7]])

Python string length (code points): 9
UTF-8 byte length: 16
Round trip: café — 北京
Whitespace split: ['Great—email', 'me', 'at', 'sam@example.org!', '#COM3110']
Characters around the dash: ['— U+2014', 'e U+0065']


In [ ]:
text.encode("utf-8")

b'caf\xc3\xa9 \xe2\x80\x94 \xe5\x8c\x97\xe4\xba\xac'

### **What the code does**

Python stores `str` values as Unicode text, while UTF-8 stores that text as a variable number of bytes. A tokenizer should receive decoded text, not raw bytes. `str.split()` only reacts to whitespace: it does not recognise the em dash, email-final exclamation mark, or hashtag as separate linguistic units. The result is deterministic, but often not task-aware.

## **Section 2 — Comparing NLTK word tokenisers**

### **Example code**

In [ ]:
sample = "I'm ready, aren't you? It costs £3.50."

outputs = {
    "str.split": sample.split(),
    "wordpunct_tokenize": wordpunct_tokenize(sample),
    "TreebankWordTokenizer": TreebankWordTokenizer().tokenize(sample),
    "word_tokenize": word_tokenize(sample),
}

for name, tokens in outputs.items():
    print(f"{name:24s} -> {tokens}")

str.split                -> ["I'm", 'ready,', "aren't", 'you?', 'It', 'costs', '£3.50.']
wordpunct_tokenize       -> ['I', "'", 'm', 'ready', ',', 'aren', "'", 't', 'you', '?', 'It', 'costs', '£', '3', '.', '50', '.']
TreebankWordTokenizer    -> ['I', "'m", 'ready', ',', 'are', "n't", 'you', '?', 'It', 'costs', '£3.50', '.']
word_tokenize            -> ['I', "'m", 'ready', ',', 'are', "n't", 'you', '?', 'It', 'costs', '£3.50', '.']


### **What the code does**

The four methods implement different boundary policies. `wordpunct_tokenize` uses a simple regular-expression split and therefore separates apostrophes and decimal points aggressively. The Treebank-style tokenizer has English-specific conventions for contractions and punctuation. `word_tokenize` applies NLTK’s recommended word tokenizer and, unless `preserve_line=True`, first invokes sentence tokenisation. There is no context-free winner: the downstream task decides whether `aren't` should be one unit or two and whether `£3.50` should be one, two, or three units.

### **Mini-test — Find tokenisation disagreements**

**Task**

Implement `compare_tokenizers(text)` using the supplied `TOKENIZERS` dictionary.

Return a dictionary with:
- `"outputs"`: a mapping from each tokenizer name to its token list.
- `"disagreements"`: a list of tokenizer-name tuples whose token lists differ.

Within each pair, names must appear in alphabetical order. Include every disagreeing pair exactly once. Compute the pairs from the outputs rather than hard-coding tokenizer names.

Replace `raise NotImplementedError` with your implementation before running the checks. The unfinished starter deliberately raises an error. Correct implementations pass silently. These public checks provide feedback without listing complete reference outputs; they are not a complete correctness test.


In [ ]:
TOKENIZERS = {
    "whitespace": str.split,
    "wordpunct": wordpunct_tokenize,
    "treebank": TreebankWordTokenizer().tokenize,
    "word_tokenize": word_tokenize,
}


from itertools import combinations


def compare_tokenizers(text):
    outputs = {
        name: tokenize(text)
        for name, tokenize in TOKENIZERS.items()
    }

    disagreements = [
        (first, second)
        for first, second in combinations(sorted(outputs), 2)
        if outputs[first] != outputs[second]
    ]

    return {
        "outputs": outputs,
        "disagreements": disagreements,
    }

# Checks

report = compare_tokenizers(
    "They won't pay £7.25; email help@example.net."
)
assert set(report["outputs"]) == set(TOKENIZERS)
assert all(isinstance(tokens, list) for tokens in report["outputs"].values())

pairs = report["disagreements"]

assert len(pairs) == len(set(pairs))
assert compare_tokenizers("") == compare_tokenizers("   ")


## **Section 3 — Sentence boundaries and character spans**

### **Example code**

In [ ]:
document = "I called Dr. Jones. I called Dr. Jones again."
sentences = sent_tokenize(document)
print("Sentences:", sentences)

span_text = "I can't log in."
treebank = TreebankWordTokenizer()
spans = list(treebank.span_tokenize(span_text))
print("Spans:", spans)
print("Recovered tokens:", [span_text[start:end] for start, end in spans])

Sentences: ['I called Dr. Jones.', 'I called Dr. Jones again.']
Spans: [(0, 1), (2, 4), (4, 7), (8, 11), (12, 14), (14, 15)]
Recovered tokens: ['I', 'ca', "n't", 'log', 'in', '.']


### **What the code does**

Punkt sentence tokenisation uses learned abbreviation and sentence-boundary evidence, so the period in `Dr.` is not automatically treated as the end of a sentence. `span_tokenize` returns half-open character offsets `(start, end)`. Offsets let a system highlight evidence in the original text and audit exactly which characters produced each token: something a list of normalised token strings cannot guarantee.

### **Mini-test — Annotate a complete document**

**Task**

Implement `annotate_document(text)` using `sent_tokenize` and `TreebankWordTokenizer`.

Return one dictionary per sentence, in document order. Each dictionary must contain:
- `"text"`: the sentence text.
- `"span"`: its half-open `(start, end)` character offsets.
- `"tokens"`: a list of `(token_text, start, end)` tuples.

**All offsets must refer to the original document**, including token offsets. Preserve the original spacing and text. Your implementation must handle repeated sentences correctly.

Replace `raise NotImplementedError` with your implementation before running the checks. The unfinished starter deliberately raises an error. Correct implementations pass silently. These public checks provide feedback without listing complete reference outputs; they are not a complete correctness test.


In [ ]:
def annotate_document(text):
    # TODO: find sentences and convert token offsets to document offsets
    raise NotImplementedError

# Checks

document = "Dr. Ahmed arrived.  We couldn't start. Dr. Ahmed waited."
annotations = annotate_document(document)
previous_end = 0
covered = set()

for item in annotations:
    start, end = item["span"]
    assert previous_end <= start < end <= len(document)
    assert document[start:end] == item["text"]
    previous_end = end
    token_end = start

    for token, token_start, token_stop in item["tokens"]:
        assert token_end <= token_start < token_stop <= end
        assert document[token_start:token_stop] == token
        covered.update(range(token_start, token_stop))
        token_end = token_stop

assert {
    index for index, character in enumerate(document)
    if not character.isspace()
} <= covered


## **Section 4 — Social-media tokenisation with `TweetTokenizer`**

### **Example code**

In [ ]:
support_message = "Email IT-support@uni.ac.uk ASAP — I can't log in 😟 #deadline"
tweet_tokenizer = TweetTokenizer(
    preserve_case=True,
    reduce_len=False,
    strip_handles=False,
)

print("TweetTokenizer:", tweet_tokenizer.tokenize(support_message))
print("word_tokenize: ", word_tokenize(support_message))

TweetTokenizer: ['Email', 'IT-support@uni.ac.uk', 'ASAP', '—', 'I', "can't", 'log', 'in', '😟', '#deadline']
word_tokenize:  ['Email', 'IT-support', '@', 'uni.ac.uk', 'ASAP', '—', 'I', 'ca', "n't", 'log', 'in', '😟', '#', 'deadline']


### **What the code does**

`TweetTokenizer` has ordered regular-expression rules for social text, including email addresses, URLs, hashtags, handles, emoticons, flags, and some joined emoji sequences. Its constructor exposes policy choices: case preservation, lengthening reduction, and handle removal. These options can change information available to a classifier, so they belong in the model specification rather than being hidden ‘cleaning’ steps.

### **Mini-test — Prepare posts for analysis**

**Task**

Implement `prepare_post(text)` using `TweetTokenizer`.

Your function must:
- Remove user handles.
- Lowercase ordinary tokens using the tokenizer's case option.
- Reduce character lengthening using the tokenizer's lengthening option.
- Replace each complete email-address token with `"<EMAIL>"`.
- Retain hashtags, emoji, and URLs as tokens.

Return the processed token list. Use the supplied email pattern to identify complete email tokens.

Replace `raise NotImplementedError` with your implementation before running the checks. The unfinished starter deliberately raises an error. Correct implementations pass silently. These public checks provide feedback without listing complete reference outputs; they are not a complete correctness test.


In [ ]:
EMAIL_PATTERN = re.compile(
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

def prepare_post(text):
    # TODO: configure TweetTokenizer and mask email tokens
    raise NotImplementedError

# Checks

first = "@Mina Email mina@example.org — soooo helpful! #NLP 😊"
second = "@Leo Email leo@example.net — soooo helpful! #NLP 😊"

# Changing only removed/masked information should not change the output.
assert prepare_post(first) == prepare_post(second)
result = prepare_post(first)
assert not any(token.startswith("@") for token in result)
assert not any(EMAIL_PATTERN.fullmatch(token) for token in result)
assert "#nlp" in result
assert "😊" in result
assert prepare_post("Email a@example.org") == prepare_post("Email b@example.net")
assert isinstance(prepare_post("Nothing to anonymise here."), list)


## **Section 5 — Designing a task-aware regex tokenizer**

### **Example code**

In [ ]:
TOKEN_PATTERN_V1 = r"""
    (?:[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})  # email
  | (?:https?://[^\s]+)                                  # simple URL
  | (?:[\#@]\w+)                                       # hashtag or handle
  | (?:[:;=8][\-^']?[()DPp/])                            # simple emoticon
  | (?:\d+(?:[.,]\d+)*)                                 # number
  | (?:\w+(?:[’']\w+)*(?:-\w+)*)                      # word
  | (?:[^\w\s])                                         # remaining symbol
"""

regex_tokenizer_v1 = RegexpTokenizer(
    TOKEN_PATTERN_V1, flags=re.VERBOSE | re.UNICODE
)

regex_example = "Email IT-support@uni.ac.uk — costs £3.50 :) #urgent"
print(regex_tokenizer_v1.tokenize(regex_example))

['Email', 'IT-support@uni.ac.uk', '—', 'costs', '£', '3.50', ':)', '#urgent']


### **What the code does**
## (Regex-Based Tokenization)

This example creates a **custom tokenizer using regular expressions**.  
Instead of simply splitting text at spaces, we define several patterns for different types of tokens such as **emails, URLs, hashtags, emoticons, numbers, words, and punctuation**.

### Token pattern

```python
TOKEN_PATTERN_V1 = r"""
    (?:[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})  # email
  | (?:https?://[^\s]+)                                  # simple URL
  | (?:[\#@]\w+)                                          # hashtag or handle
  | (?:[;:=8][\-^']?[(\)DPp/])                            # simple emoticon
  | (?:\d+(?:[.,]\d+)*)                                   # number
  | (?:\w+(?:[’']\w+)*(?:-\w+)*)                          # word
  | (?:[^\w\s])                                           # remaining symbol
"""
```

The `|` symbol means **OR**, so the tokenizer tries to match any of these token types.

Importantly, the alternatives are checked **from left to right**.  
More specific patterns such as emails and URLs are therefore placed before more general patterns such as words and symbols.

---

### 1. Email addresses

```text
[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}
```

This pattern recognises email addresses.

For example:

```text
IT-support@uni.ac.uk
john.smith@gmail.com
```

The main components are:

- `[A-Za-z0-9._%+-]+` → username part
- `@` → the `@` symbol
- `[A-Za-z0-9.-]+` → domain name
- `\.` → literal dot
- `[A-Za-z]{2,}` → domain suffix such as `uk`, `com`, or `org`

So:

```text
IT-support@uni.ac.uk
```

is kept as **one token** rather than being split around `-`, `@`, or `.`.

---

### 2. URLs

```text
https?://[^\s]+
```

This recognises simple web URLs.

For example:

```text
https://www.sheffield.ac.uk
http://example.com
```

Here:

- `http` matches the beginning
- `s?` means the `s` is optional, allowing both `http` and `https`
- `://` matches the URL separator
- `[^\s]+` means one or more characters that are **not whitespace**

---

### 3. Hashtags and user handles

```text
[\#@]\w+
```

This matches strings beginning with either `#` or `@`.

For example:

```text
#urgent
#NLP
@OpenAI
```

The `\w+` part means one or more word characters.

---

### 4. Simple emoticons

```text
[;:=8][\-^']?[(\)DPp/]
```

This pattern recognises simple text-based emoticons such as:

```text
:)
:-)
:D
;)
:P
```

It roughly consists of:

1. an eye character such as `:`, `;`, `=` or `8`
2. an optional nose such as `-`
3. a mouth character such as `)`, `(`, `D`, `P`, etc.

This allows an emoticon such as

```text
:)
```

to remain a single token.

---

### 5. Numbers

```text
\d+(?:[.,]\d+)*
```

This recognises numbers containing optional decimal or separator characters.

For example:

```text
3
3.50
1,000
12.75
```

The first `\d+` matches one or more digits.

The second part:

```text
(?:[.,]\d+)*
```

allows additional groups beginning with `.` or `,`.

Therefore:

```text
3.50
```

is treated as one token rather than:

```text
3
.
50
```

---

### 6. Words

```text
\w+(?:[’']\w+)*(?:-\w+)*
```

This matches ordinary words while also supporting **apostrophes and hyphens**.

Examples include:

```text
hello
don't
student's
state-of-the-art
```

The first:

```text
\w+
```

matches the main word.

The second part:

```text
(?:[’']\w+)*
```

allows apostrophes, such as:

```text
don't
student's
```

The final part:

```text
(?:-\w+)*
```

allows hyphenated words, such as:

```text
state-of-the-art
```

---

### 7. Remaining symbols

```text
[^\w\s]
```

This catches characters that are neither:

- word characters (`\w`)
- whitespace (`\s`)

Examples include:

```text
£
!
?
–
,
.
```

This acts as a **fallback rule** for punctuation and other symbols that were not matched by an earlier pattern.

---

## Creating the tokenizer

```python
regex_tokenizer_v1 = RegexpTokenizer(
    TOKEN_PATTERN_V1,
    flags=re.VERBOSE | re.UNICODE
)
```

`RegexpTokenizer` applies our regular-expression rules to the input text.

Two flags are used:

### `re.VERBOSE`

This allows the regular expression to be written across multiple lines and lets us include comments such as:

```python
# email
# number
# word
```

This makes a complicated regular expression much easier to read.

### `re.UNICODE`

This enables Unicode-aware handling of characters, which is particularly useful when processing text containing non-ASCII characters.

---

## Example

```python
regex_example = "Email IT-support@uni.ac.uk – costs £3.50 :) #urgent"

print(regex_tokenizer_v1.tokenize(regex_example))
```

The tokenizer produces approximately:

```text
[
    'Email',
    'IT-support@uni.ac.uk',
    '–',
    'costs',
    '£',
    '3.50',
    ':)',
    '#urgent'
]
```

Notice that meaningful structures are preserved:

| Input | Tokenisation |
|---|---|
| `Email` | `Email` |
| `IT-support@uni.ac.uk` | kept as one email token |
| `–` | punctuation/symbol |
| `costs` | word |
| `£` | symbol |
| `3.50` | kept as one number |
| `:)` | kept as one emoticon |
| `#urgent` | kept as one hashtag |

---

## Why not just use `text.split()`?

A simple whitespace tokenizer would split only when it sees spaces:

```python
text.split()
```

This would give:

```text
['Email', 'IT-support@uni.ac.uk', '–', 'costs', '£3.50', ':)', '#urgent']
```

The regex tokenizer is more flexible because it understands the **structure of different token types**.

For example, it separates:

```text
£3.50
```

into:

```text
£
3.50
```

while still keeping the decimal number `3.50` intact.

---

## Key idea

A regex tokenizer lets us explicitly define:

> **What should count as a token?**

This makes tokenisation **controllable and interpretable**.

However, these are manually written rules, so they may not cover every possible case. Modern NLP models therefore often use learned subword tokenizers such as **BPE, WordPiece, or SentencePiece**, which we can compare with this rule-based approach later.

## **Answer — Find tokenisation disagreements (Sec.1)**


In [ ]:
TOKENIZERS = {
    "whitespace": str.split,
    "wordpunct": wordpunct_tokenize,
    "treebank": TreebankWordTokenizer().tokenize,
    "word_tokenize": word_tokenize,
}

from itertools import combinations

def compare_tokenizers(text):
    outputs = {
        name: tokenize(text)
        for name, tokenize in TOKENIZERS.items()
    }
    disagreements = [
        (first, second)
        for first, second in combinations(sorted(outputs), 2)
        if outputs[first] != outputs[second]
    ]
    return {"outputs": outputs, "disagreements": disagreements}

report = compare_tokenizers(
    "They won't pay £7.25; email help@example.net."
)
assert set(report["outputs"]) == set(TOKENIZERS)
assert all(isinstance(tokens, list) for tokens in report["outputs"].values())

pairs = report["disagreements"]
assert all(
    first in TOKENIZERS and second in TOKENIZERS and first < second
    for first, second in pairs
)
assert len(pairs) == len(set(pairs))
assert compare_tokenizers("") == compare_tokenizers("   ")


## **Answer — Annotate a complete document (Sec. 3)**

Reference solution for the additional mini-test. Attempt the exercise before reading this code.


In [ ]:
def annotate_document(text):
    annotations = []
    cursor = 0
    tokenizer = TreebankWordTokenizer()

    for sentence in sent_tokenize(text):
        # Search from the previous sentence's end to handle repetition.
        start = text.index(sentence, cursor)
        end = start + len(sentence)
        tokens = [
            (text[start + a:start + b], start + a, start + b)
            for a, b in tokenizer.span_tokenize(sentence)
        ]
        annotations.append({
            "text": sentence,
            "span": (start, end),
            "tokens": tokens,
        })
        cursor = end

    return annotations

document = "Dr. Ahmed arrived.  We couldn't start. Dr. Ahmed waited."
annotations = annotate_document(document)
previous_end = 0
covered = set()

for item in annotations:
    start, end = item["span"]
    assert previous_end <= start < end <= len(document)
    assert document[start:end] == item["text"]
    previous_end = end
    token_end = start

    for token, token_start, token_stop in item["tokens"]:
        assert token_end <= token_start < token_stop <= end
        assert document[token_start:token_stop] == token
        covered.update(range(token_start, token_stop))
        token_end = token_stop

assert {
    index for index, character in enumerate(document)
    if not character.isspace()
} <= covered


## **Answer — Prepare posts for analysis (Sec. 4)**

Reference solution for the additional mini-test in Section 4. Attempt the exercise before reading this code.


In [ ]:
EMAIL_PATTERN = re.compile(
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

def prepare_post(text):
    tokenizer = TweetTokenizer(
        strip_handles=True,
        preserve_case=False,
        reduce_len=True,
    )
    return [
        "<EMAIL>" if EMAIL_PATTERN.fullmatch(token) else token
        for token in tokenizer.tokenize(text)
    ]

first = "@Mina Email mina@example.org — soooo helpful! #NLP 😊"
second = "@Leo Email leo@example.net — soooo helpful! #NLP 😊"

# Changing only removed/masked information should not change the output.
assert prepare_post(first) == prepare_post(second)
result = prepare_post(first)
assert not any(token.startswith("@") for token in result)
assert not any(EMAIL_PATTERN.fullmatch(token) for token in result)
assert "#nlp" in result
assert "😊" in result
assert prepare_post("Email a@example.org") == prepare_post("Email b@example.net")
assert isinstance(prepare_post("Nothing to anonymise here."), list)


---
## **References and follow-up**

- *W1 Introduction*, visible slides 26–52.
- [NLTK tokenization API](https://www.nltk.org/api/nltk.tokenize.html)
- [NLTK tokenization examples](https://www.nltk.org/howto/tokenize.html)
- [NLTK Punkt sentence tokenizer](https://www.nltk.org/api/nltk.tokenize.punkt.html)
- Sennrich, Haddow, and Birch (2016), *Neural Machine Translation of Rare Words with Subword Units*.

**Optional extension:** compare these word-like tokens with the supplied subword tokenizer of an open language model. Inspect where one NLTK token becomes several subword IDs and how the result changes across languages.